In [ ]:
import glob
import re
import os

import numpy as np
from pathlib import Path

from pymor.basic import *
from pymor.core.pickle import load

from RBInvParam.problems.elasticity.build import build_InstationaryModelIP

set_log_levels({
    'pymor' : 'WARN'
})

set_defaults({})


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams.update({
    "text.usetex": True,          # set to True if you have LaTeX installed
    "font.family": "cm",
    #"font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 14,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 150,
})

COLOR_CYCLE = [
    "#4C72B0",  # muted blue
    "#DD8452",  # muted orange
    "#55A868",  # muted green
    "#C44E52",  # muted red
    "#8172B3",  # muted purple
    "#937860",  # muted brown
    "#DA8BC3",  # muted pink
    "#8C8C8C",  # muted gray
    "#CCB974",  # muted yellow
    "#64B5CD",  # muted cyan
    "#4C9A8A",  # muted teal
    "#A17C6B",  # muted clay
]

MARKERS = [
    "o",   # circle
    "s",   # square
    "^",   # triangle up
    "v",   # triangle down
    "D",   # diamond
    "P",   # plus-filled
    "X",   # x-filled
    "*",   # star
    "<",   # triangle left
    ">",   # triangle right
    "h",   # hexagon1
    "H",   # hexagon2
]

# COLOR_CYCLE = [
#     "#000000",  # black
#     "#E69F00",  # orange
#     "#56B4E9",  # sky blue
#     "#009E73",  # bluish green
#     "#F0E442",  # yellow
#     "#0072B2",  # blue
#     "#D55E00",  # vermillion
#     "#CC79A7",  # reddish purple
# ]

# MARKERS = [
#     "o", 
#     "s", 
#     "^", 
#     "v", 
#     "D", 
#     "P", 
#     "X", 
#     "*"
# ]

In [ ]:
from typing import Dict, Tuple, Optional

def get_last_file(path: Path) -> Path | None:
    # --- Step 1: Look for final files first ---
    final_candidates = [
        path / "TR_IRGNM_final.pkl",
        path / "FOM_IRGNM_final.pkl"
    ]
    
    for final_file in final_candidates:
        if final_file.exists():
            return final_file.name  # Return immediately if found
    
    # --- Step 2: If no final file exists, find the highest index file ---
    files = glob.glob(os.path.join(path, "TR_IRGNM_*.pkl"))
    files += glob.glob(os.path.join(path, "FOM_IRGNM_*.pkl"))

    indexed_files = []
    for f in files:
        match = re.search(r'(?:TR|FOM)_IRGNM_(\d+)\.pkl$', os.path.basename(f))
        if match:
            idx = int(match.group(1))
            indexed_files.append((idx, f))

    if indexed_files:
        _, max_file = max(indexed_files, key=lambda x: x[0])
        return Path(max_file).name

    print("No matching IRGNM result files found.")
    return None

def filter_and_reorder(d: Dict, pattern: str = r'.*FOM.*') -> Tuple[Dict, Optional[str]]:
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    non_matching = [k for k in d if k not in matching]

    # Sort keys alphabetically within each group
    non_matching_sorted = sorted(non_matching)
    matching_sorted = sorted(matching)

    # Build the reordered dict: alphabetically sorted non-matching first, then matching ones
    reordered = {k: d[k] for k in non_matching_sorted}
    reordered.update({k: d[k] for k in matching_sorted})

    # Return reordered dict and the single matching key (if exactly one match)
    if len(matching_sorted) == 1:
        return reordered, matching_sorted[0]
    else:
        return reordered, None

In [ ]:
import glob
import os
import re
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pymor.core.pickle import load


# ============================================================
# Style (paper-ready, readable after scaling)
# ============================================================
plt.style.use('default')
plt.rcParams.update({
    "text.usetex": True,          # set to True if you have LaTeX installed
    "font.family": "cm",
    #"font.family": "serif",
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 22,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "figure.dpi": 150,
})


COLOR_CYCLE = [
    "#4C72B0",
    "#DD8452",
    "#55A868",
    "#C44E52",
    "#8172B3",
    "#937860",
    "#DA8BC3",
    "#8C8C8C",
    "#CCB974",
    "#64B5CD",
    "#4C9A8A",
    "#A17C6B",
]

LOWER_ROW_BLUES = [
    "#4C72B0",
    "#2F5D9B",
    "#6F92C6",
    "#8FB1D8",
]


MAX_OUTER_ITER = 55
MAX_INNER_ITER = 300
OFFSET = 10
iterval = 20
xticks = [0 + i * iterval for i in range(int(MAX_INNER_ITER/iterval))]


# ============================================================
# Paths / setup
# ============================================================
# SAVE_PATH = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/figs')
# WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')

SAVE_PATH = Path('/home/dealii/workdir/figs')
WORK_DIR = Path('/home/dealii/workdir/experiments')

subset = 'elasticity'
obs_op = 'sensors'

main_data_dir = WORK_DIR / 'elasticity_alu_non_normalize'  / 'elasticity_alu_noise_level'
error_data_dir = WORK_DIR / 'elasticity_alu_non_normalize'  / 'elasticity_alu_noise_level_0_error'

representative_experiment = 'elasticity_alu_TR_sensors_noise_level_0.0'

# ============================================================
# Helpers
# ============================================================
def get_last_file(path: Path) -> Path | None:
    final_candidates = [
        path / "TR_IRGNM_final.pkl",
        path / "FOM_IRGNM_final.pkl"
    ]
    for final_file in final_candidates:
        if final_file.exists():
            return final_file.name

    files = glob.glob(os.path.join(path, "TR_IRGNM_*.pkl"))
    files += glob.glob(os.path.join(path, "FOM_IRGNM_*.pkl"))

    indexed_files = []
    for f in files:
        match = re.search(r'(?:TR|FOM)_IRGNM_(\d+)\.pkl$', os.path.basename(f))
        if match:
            indexed_files.append((int(match.group(1)), f))

    if indexed_files:
        _, max_file = max(indexed_files, key=lambda x: x[0])
        return Path(max_file).name

    print(f"No matching IRGNM result files found in {path}.")
    return None


def filter_and_reorder(d, pattern=r'.*FOM.*'):
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    non_matching = [k for k in d if k not in matching]

    non_matching_sorted = sorted(non_matching)
    matching_sorted = sorted(matching)

    reordered = {k: d[k] for k in non_matching_sorted}
    reordered.update({k: d[k] for k in matching_sorted})

    if len(matching_sorted) == 1:
        return reordered, matching_sorted[0]
    return reordered, None


def load_experiment_bundle(data_dir_path: Path, obs_op: str):
    experiment_names = []
    pattern = re.compile(rf'^.*_{obs_op}')

    for p in data_dir_path.iterdir():
        if p.is_dir():
            name = p.name
            if pattern.match(name) and not name.endswith('every_5th'):
                experiment_names.append(name)

    data_paths = [data_dir_path / experiment_name for experiment_name in experiment_names]
    file_names = [get_last_file(data_path) for data_path in data_paths]

    setup = None
    data = {}
    optimizer_parameters = {}

    for data_path, file_name in zip(data_paths, file_names):
        if file_name is None:
            continue

        try:
            with open(data_path / file_name, 'rb') as file:
                data_ = load(file)
            data[str(data_path.name)] = data_
        except TypeError:
            print(f"Can not find dumps for {data_path}")
        except Exception:
            print(f"Can not open {data_path / file_name}")

        if setup is None:
            with open(data_path / 'setup.pkl', 'rb') as file:
                setup = load(file)

        optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
        with open(optimizer_parameter_path, 'rb') as file:
            optimizer_parameter = load(file)

        optimizer_parameters[str(data_path.name)] = optimizer_parameter

    data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')
    return setup, data, optimizer_parameters, FOM_key


def parse_experiment_name(name):
    method = "FOM" if "_FOM_" in name else "TR" if "_TR_" in name else "UNKNOWN"
    m = re.search(r"noise_level_([0-9.]+)$", name)
    noise = float(m.group(1)) if m else None
    return method, noise


def compute_Js(experiment_data):
    Js = []
    try:
        inner = experiment_data.get('inner_loop_statistics', None)
        if inner:
            inner = inner[:MAX_OUTER_ITER]
            for stat in inner:
                Js += stat['J']
        else:
            Js += experiment_data['J']
    except Exception:
        Js += experiment_data['J']
    return np.asarray(Js)


def get_runtime_array(experiment_data, Js):
    try:
        times = np.array([0.0])
        times = np.append(times, experiment_data['total_runtime'])
    except Exception:
        times = np.array([0.0])
        times = np.append(times, experiment_data['time_steps'])

    if len(Js) != len(times):
        times = times[:-1]
        times = times[1:]
        if len(times) > 0:
            times[0] = 0.0

    n = min(len(times), len(Js))
    return times[:n], Js[:n]


def merge_error_series(experiment_data):
    merged = {}
    stats = experiment_data.get('inner_loop_statistics', [])
    for entry in stats:
        for key, value in entry.get('errors', {}).items():
            merged.setdefault(key, []).extend(value)
        merged.setdefault('J', []).extend(entry['J'])
        merged.setdefault('delta_J', []).extend(np.abs(
            np.array(entry['J'][1:]) -
            np.array(entry['J'][:-1])
        ))

    for key in merged:
        merged[key] = np.asarray(merged[key])
    return merged


def candidate_gradient_keys(merged_errors):
    keys = list(merged_errors.keys())

    priority = [
        "J",
        "err_J",
        "rel_err_J",
    ]

    selected = [k for k in priority if k in keys]

    if len(selected) < 2:
        fallback = []
        for k in keys:
            kl = k.lower()
            if ("grad" in kl or "dj" in kl) and k not in selected:
                fallback.append(k)
        selected += fallback

    return selected[:3]


def pretty_error_label(key):
    custom = {
        "J": r"$J_r$",
        "err_J": r"$\Delta^J_{\mathrm{real}}$",
        "rel_err_J": r"$\Delta^J_{\mathrm{real}} / J_r$",
    }

    if key in custom:
        return custom[key]
    return r"$\mathrm{{{key.replace('_', r'\_')}}}$"


def get_style(name):
    method, noise = parse_experiment_name(name)
    color = color_map.get(noise, "#333333")

    if method == "TR":
        return dict(
            color=color,
            linestyle="-",
            marker="o",
            linewidth=2.0,
            markersize=4.6,
            markerfacecolor=color,
            markeredgecolor=color,
            markeredgewidth=0.7,
        )
    elif method == "FOM":
        return dict(
            color=color,
            linestyle="--",
            marker="s",
            linewidth=1.8,
            markersize=4.6,
            markerfacecolor=(1, 1, 1, 0.75),
            markeredgecolor=color,
            markeredgewidth=0.7,
        )
    else:
        return dict(
            color="#333333",
            linestyle=":",
            marker="x",
            linewidth=1.8,
            markersize=4.6,
            markerfacecolor="#333333",
            markeredgecolor="#333333",
            markeredgewidth=0.7,
        )


def get_delta_zero_style(idx):
    colors = LOWER_ROW_BLUES
    #markers = ["o", "^", "D"]
    markers = [None]
    linestyles = ["-", "--", "-.", ":"]
    c = colors[idx % len(colors)]
    m = markers[idx % len(markers)]
    ls = linestyles[idx % len(linestyles)]

    return dict(
        color=c,
        linestyle=ls,
        marker=m,
        linewidth=2.0 if idx == 0 else 1.8,
        markersize=4.5,
        markerfacecolor=(1, 1, 1, 0.85) if idx > 0 else c,
        markeredgecolor=c,
        markeredgewidth=0.7,
    )


def extract_noise_level(name: str):
    m = re.search(r"noise_level_([0-9.eE+-]+)$", name)
    return float(m.group(1)) if m else None


def format_noise_scientific(x):
    if x == 0:
        return r"$\delta_{\mathrm{rel.}} = 0$"
    exp = int(np.floor(np.log10(abs(x))))
    mant = x / (10**exp)
    return rf"$\delta_{{\mathrm{{rel.}}}} = {mant:.1f}\cdot 10^{{{exp}}}$"


# ============================================================
# Load both datasets
# ============================================================
setup_main, data_main, optimizer_parameters_main, FOM_key_main = load_experiment_bundle(main_data_dir, obs_op)
setup_err, data_err, optimizer_parameters_err, FOM_key_err = load_experiment_bundle(error_data_dir, obs_op)

max_noise_level = 1.0 * 1e-2

data_main = {
    k: v for k, v in data_main.items()
    if (extract_noise_level(k) is not None and extract_noise_level(k) <= max_noise_level)
}

optimizer_parameters_main = {
    k: v for k, v in optimizer_parameters_main.items()
    if k in data_main
}

data_err = {
    k: v for k, v in data_err.items()
    if (extract_noise_level(k) is not None and extract_noise_level(k) <= max_noise_level)
}

optimizer_parameters_err = {
    k: v for k, v in optimizer_parameters_err.items()
    if k in data_err
}

if representative_experiment not in data_err:
    available = sorted(data_err.keys())
    representative_experiment = available[0]
    print(f"Representative experiment not found. Falling back to: {representative_experiment}")

# ============================================================
# Shared ordering / color mapping across all panels
# ============================================================
all_names = sorted(set(data_main.keys()) | set(data_err.keys()))
all_noise_levels = sorted({
    parse_experiment_name(name)[1]
    for name in all_names
    if parse_experiment_name(name)[1] is not None
})

color_map = {
    nl: COLOR_CYCLE[i % len(COLOR_CYCLE)]
    for i, nl in enumerate(all_noise_levels)
}

desired_order_main = sorted(
    data_main.keys(),
    key=lambda name: (
        parse_experiment_name(name)[1] if parse_experiment_name(name)[1] is not None else np.inf,
        0 if parse_experiment_name(name)[0] == "TR" else 1,
        name,
    )
)

desired_order_err = sorted(
    data_err.keys(),
    key=lambda name: (
        parse_experiment_name(name)[1] if parse_experiment_name(name)[1] is not None else np.inf,
        0 if parse_experiment_name(name)[0] == "TR" else 1,
        name,
    )
)

# ============================================================
# Build figures
# ============================================================
fig_top, axs_top = plt.subplots(1, 2, figsize=(16.5, 6.2))
ax_iter, ax_time = axs_top

fig_bottom, axs_bottom = plt.subplots(1, 2, figsize=(16.5, 6.2))
ax_grad, ax_dim = axs_bottom

# ------------------------------------------------------------
# Top-left: objective vs total iterations
# ------------------------------------------------------------
for name in desired_order_main:
    Js = compute_Js(data_main[name])[:MAX_INNER_ITER]
    style = get_style(name).copy()
    style.update({
        "linewidth": 2.2,
        "markersize": 5.2,
        "markeredgewidth": 0.8,
    })
    ax_iter.plot(
        np.arange(len(Js)),
        Js,
        **style
    )

ax_iter.set_xlabel("Total iterations")
ax_iter.set_ylabel(r"$J_\ast(q_\ast^{(i)})$")
ax_iter.set_yscale("log")
ax_iter.set_title(r"(a) $J_\ast(q_\ast^{(i)})$ vs.\ iterations", pad=8)
ax_iter.set_xlim([-OFFSET, MAX_INNER_ITER+OFFSET])
ax_iter.set_xticks(xticks)

# ------------------------------------------------------------
# Top-right: objective vs runtime
# ------------------------------------------------------------
for name in desired_order_main:
    Js = np.asarray(data_main[name]["J"])[:MAX_INNER_ITER]
    times, Js = get_runtime_array(data_main[name], Js)
    style = get_style(name).copy()
    style.update({
        "linewidth": 2.2,
        "markersize": 5.2,
        "markeredgewidth": 0.8,
    })
    ax_time.plot(
        times,
        Js,
        **style
    )

ax_time.set_xlabel("Runtime [s]")
ax_time.set_ylabel(r"$J_\ast(q_\ast^{(i)})$")
ax_time.set_yscale("log")
ax_time.set_title(r"(b) $J_\ast(q_\ast^{(i)})$ vs.\ runtime", pad=8)

# ------------------------------------------------------------
# Legend for top figure (below both plots)
# ------------------------------------------------------------
method_handles = [
    Line2D(
        [0], [0],
        color="black",
        linestyle="-",
        marker="o",
        linewidth=2.2,
        markersize=5.2,
        markerfacecolor="black",
        markeredgecolor="black",
        markeredgewidth=0.8,
        label="TR"
    ),
    Line2D(
        [0], [0],
        color="black",
        linestyle="--",
        marker="s",
        linewidth=2.0,
        markersize=5.2,
        markerfacecolor=(1, 1, 1, 0.75),
        markeredgecolor="black",
        markeredgewidth=0.8,
        label="FOM"
    ),
]

noise_handles = [
    Line2D(
        [0], [0],
        color=color_map[nl],
        linestyle="-",
        linewidth=2.6,
        label=format_noise_scientific(nl)
    )
    for nl in all_noise_levels
]

combined_handles = method_handles + noise_handles
combined_labels = [h.get_label() for h in combined_handles]

fig_top.legend(
    handles=combined_handles,
    labels=combined_labels,
    loc="lower center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, -0.05),
    handlelength=2.6,
    handletextpad=0.6,
    columnspacing=1.6,
)

# ------------------------------------------------------------
# Bottom-left: diagnostics for one representative delta=0 run
# ------------------------------------------------------------
rep_data = data_err[representative_experiment]
merged_errors = merge_error_series(rep_data)
grad_keys = candidate_gradient_keys(merged_errors)

if len(grad_keys) == 0:
    fallback_keys = [k for k in ["J", "err_J", "rel_err_J"] if k in merged_errors]
    grad_keys = fallback_keys[:2]

for idx, key in enumerate(grad_keys):
    values = merged_errors[key][:MAX_INNER_ITER]
    x = np.arange(len(values))
    style = get_delta_zero_style(idx).copy()
    style.update({
        "linewidth": 2.2 if idx == 0 else 2.0,
        "markersize": 5.0,
        "markeredgewidth": 0.8,
    })

    ax_grad.plot(
        x,
        values,
        **style,
        label=pretty_error_label(key),
    )

ax_grad.set_xlabel("Total iterations")
ax_grad.set_ylabel("Error")
ax_grad.set_yscale("log")
ax_grad.set_title(r"(a) Objective error $(\delta_{\mathrm{rel.}} = 0)$", pad=8)
ax_grad.set_xlim([-OFFSET, MAX_INNER_ITER+OFFSET])
ax_grad.set_xticks(xticks)

# ------------------------------------------------------------
# Bottom-right: reduced state-space dimension vs outer iterations
# ------------------------------------------------------------
delta_zero_names = [
    name for name in desired_order_err
    if parse_experiment_name(name)[1] == 0.0
]

delta_zero_color_map = {
    name: LOWER_ROW_BLUES[i % len(LOWER_ROW_BLUES)]
    for i, name in enumerate(delta_zero_names)
}

def get_delta_zero_dim_style(name):
    method, noise = parse_experiment_name(name)
    base_color = delta_zero_color_map.get(name, LOWER_ROW_BLUES[0])

    if method == "TR":
        return dict(
            color=base_color,
            linestyle="-",
            marker="o",
            linewidth=2.2,
            markersize=4.8,
            markerfacecolor=base_color,
            markeredgecolor=base_color,
            markeredgewidth=0.8,
        )
    elif method == "FOM":
        return dict(
            color=base_color,
            linestyle="--",
            marker="s",
            linewidth=2.0,
            markersize=4.8,
            markerfacecolor=(1, 1, 1, 0.85),
            markeredgecolor=base_color,
            markeredgewidth=0.8,
        )
    else:
        return dict(
            color=base_color,
            linestyle=":",
            marker="x",
            linewidth=2.0,
            markersize=4.8,
            markerfacecolor=base_color,
            markeredgecolor=base_color,
            markeredgewidth=0.8,
        )


max_dim = 0
dim_handles = []

for idx, name in enumerate(delta_zero_names):
    if "dim_V_r" not in data_err[name]:
        continue
    dim_V_r = np.asarray(data_err[name]["dim_V_r"])[:MAX_OUTER_ITER]
    if len(dim_V_r) == 0:
        continue

    # unique matching style for dim V_r
    dim_color = "#4C72B0"
    dim_linestyle = "-"
    dim_marker = "D"
    dim_linewidth = 2.4
    dim_markersize = 5.4
    dim_markerfacecolor = "white"
    dim_markeredgecolor = dim_color
    dim_markeredgewidth = 1.0

    ax_dim.step(
        np.arange(len(dim_V_r))[:MAX_INNER_ITER],
        dim_V_r,
        where="post",
        color=dim_color,
        linestyle=dim_linestyle,
        linewidth=dim_linewidth,
        alpha=0.98,
    )
    ax_dim.plot(
        np.arange(len(dim_V_r)),
        dim_V_r,
        linestyle="None",
        marker=dim_marker,
        color=dim_color,
        markersize=dim_markersize,
        markerfacecolor=dim_markerfacecolor,
        markeredgecolor=dim_markeredgecolor,
        markeredgewidth=dim_markeredgewidth,
    )

    # add legend handle only once
    if not dim_handles:
        dim_handles.append(
            Line2D(
                [0], [0],
                color=dim_color,
                linestyle=dim_linestyle,
                marker=dim_marker,
                linewidth=dim_linewidth,
                markersize=dim_markersize,
                markerfacecolor=dim_markerfacecolor,
                markeredgecolor=dim_markeredgecolor,
                markeredgewidth=dim_markeredgewidth,
                label=r"$\dim V_r$",
            )
        )

    max_dim = max(max_dim, np.max(dim_V_r))

ax_dim.set_xlabel("Outer iterations")
ax_dim.set_ylabel(r"$\dim V_r$")
ax_dim.set_title(r"(b) Reduced state-space dimension $(\delta_{\mathrm{rel.}} = 0)$", pad=8)
if max_dim > 0:
    ax_dim.set_ylim([0, 1.05 * max_dim])

# ------------------------------------------------------------
# Legend for bottom figure (below both plots)
# ------------------------------------------------------------
grad_legend_handles, grad_legend_labels = ax_grad.get_legend_handles_labels()
bottom_handles = grad_legend_handles + dim_handles
bottom_labels = grad_legend_labels+ [h.get_label() for h in dim_handles]

if bottom_handles:
    fig_bottom.legend(
        handles=bottom_handles,
        labels=bottom_labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, -0.05),
        handlelength=2.6,
        handletextpad=0.6,
        columnspacing=1.6,
    )

# ------------------------------------------------------------
# Common axes styling
# ------------------------------------------------------------
for ax in (ax_iter, ax_time, ax_grad, ax_dim):
    ax.grid(True, which="major", linestyle="-", alpha=0.30)
    ax.grid(True, which="minor", linestyle=":", alpha=0.20)
    ax.tick_params(axis="both", which="major", length=5)
    ax.tick_params(axis="both", which="minor", length=3)

for ax in (ax_iter, ax_grad, ax_dim):
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

ax_dim.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------
fig_top.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.22,
    top=0.90,
    wspace=0.28,
)

fig_bottom.subplots_adjust(
    left=0.08,
    right=0.98,
    bottom=0.22,
    top=0.90,
    wspace=0.28,
)

# ------------------------------------------------------------
# Save / show
# ------------------------------------------------------------
fig_top.savefig(SAVE_PATH / "noise_level_decays.pdf", bbox_inches="tight")
fig_bottom.savefig(SAVE_PATH / "noise_level_error.pdf", bbox_inches="tight")

plt.show()